In [7]:

import datetime
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentinelhub import (
    CRS,
    BBox,
    DataCollection,
    DownloadRequest,
    MimeType,
    MosaickingOrder,
    SentinelHubDownloadClient,
    SentinelHubRequest,
    bbox_to_dimensions,
)

from utils import plot_image

from sentinelhub import SHConfig, SentinelHubRequest
from tqdm import tqdm
config = SHConfig()
config.sh_client_id = '9f125395-0bd9-483e-b22c-f4d53df74900'
config.sh_client_secret = 'rU8a7nkqjKtmW9Uqc1oMJotKYVuCUbr3'


from sentinelhub import SHConfig



In [2]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [20]:
from curses import window


evalscript_true_color = """
    //VERSION=3

    function setup() {
        return {
            input: [{
                bands: ["B02", "B03", "B04", "CLM"]
            }],
            output: {
                bands: 4
            }
        };
    }

    function evaluatePixel(sample) {
        return [sample.B04, sample.B03, sample.B02, sample.CLM];
    }
"""
# download data for the points of interest

date_to_collects = [[21,3],
                    [7,4],
                    [21,4],
                    [7,5],
                    [21,5],
                    [7,6],
                    [7,7]
                    ]
years_range = range(2016, 2026)
window_radius = 3
def get_ranges_to_collect(year, month, day, window_radius):
    date = datetime.date(year, month, day)
    date_start = date - datetime.timedelta(days=window_radius)
    date_end = date + datetime.timedelta(days=window_radius)
    return date_start, date_end
    


def download_data_for_point(lon, lat, date_start, date_end, save_path, delta_lat = 0.006734416666667187, delta_lon = 0.017567916666666778):
    # Define the bounding box around the point of interest
    bbox = BBox(bbox=[lon - delta_lon, lat - delta_lat, lon + delta_lon, lat + delta_lat], crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox, resolution=10)
    # Create a SentinelHubRequest to download the data
    req = SentinelHubRequest(
        evalscript=evalscript_true_color,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L1C,
                time_interval=(date_start, date_end),  # Your date range
                mosaicking_order=MosaickingOrder.LEAST_CC,
            )
        ],
        responses=[
            SentinelHubRequest.output_response("default", MimeType.PNG)
        ],
        bbox=bbox,
        size=size,
        config=config
    )

    img = req.get_data()[0]
    # calculate the percentage of cloud cover in the image
    cloud_cover = np.mean(img[:, :, 3])/255
    too_much_cloud = cloud_cover > 0.1
    if too_much_cloud:
        print(f"Image for point ({lon}, {lat}) and date range ({date_start} - {date_end}) has too much cloud cover ({cloud_cover:.2%}).")
        # add a C_marker to save path
        save_path = save_path.replace(".png", "_C.png")
    # img = img[:, :, :3]  # Keep only RGB channel
    if img.shape[0]<128 or img.shape[1]<128:
        print(f"Image for point ({lon}, {lat}) and date range ({date_start} - {date_end}) is smaller than 128x128 pixels.")
    # revrse the image last band
    img[:, :, 3] = 255 - img[:, :, 3]
    # store the image
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    import matplotlib.pyplot as plt
    plt.imsave(save_path, img)
    return img

def save_path_for_point(index,lon, lat, year, month,day, date_start, date_end):
    # Extract month from date_start (format is YYYY-MM-DD)
    return f"data/{index:03d}_{lon}_{lat}/{year}_{month:02d}_{day:02d}.png"

def main_download(years_range=years_range, date_to_collects=date_to_collects):
    points_of_interest = pd.read_csv("data/points_of_interest_10m.csv").values
    # Calculate grid dimensions

    for index, point in tqdm(enumerate(points_of_interest), total=len(points_of_interest)):
        for year in years_range:
            for day, month in date_to_collects:
                date_start, date_end = get_ranges_to_collect(year, month, day, window_radius)
                save_path = save_path_for_point(index, point[0], point[1], year, month, day, date_start, date_end)
                img = download_data_for_point(point[0], point[1], date_start, date_end, save_path)
main_download()


  0%|          | 0/36 [00:00<?, ?it/s]

Image for point (24.13044, 65.918934) and date range (2016-04-04 - 2016-04-10) has too much cloud cover (100.00%).
Image for point (24.13044, 65.918934) and date range (2016-04-18 - 2016-04-24) has too much cloud cover (10.08%).
Image for point (24.13044, 65.918934) and date range (2016-05-18 - 2016-05-24) has too much cloud cover (100.00%).
Image for point (24.13044, 65.918934) and date range (2016-06-04 - 2016-06-10) has too much cloud cover (100.00%).
Image for point (24.13044, 65.918934) and date range (2016-07-04 - 2016-07-10) has too much cloud cover (100.00%).
Image for point (24.13044, 65.918934) and date range (2017-03-18 - 2017-03-24) has too much cloud cover (92.07%).
Image for point (24.13044, 65.918934) and date range (2017-04-04 - 2017-04-10) has too much cloud cover (100.00%).
Image for point (24.13044, 65.918934) and date range (2017-05-04 - 2017-05-10) has too much cloud cover (100.00%).
Image for point (24.13044, 65.918934) and date range (2017-06-04 - 2017-06-10) has

  3%|▎         | 1/36 [00:47<27:44, 47.56s/it]

Image for point (24.090357, 65.925954) and date range (2016-04-04 - 2016-04-10) has too much cloud cover (100.00%).
Image for point (24.090357, 65.925954) and date range (2016-04-18 - 2016-04-24) has too much cloud cover (43.95%).
Image for point (24.090357, 65.925954) and date range (2016-05-18 - 2016-05-24) has too much cloud cover (100.00%).
Image for point (24.090357, 65.925954) and date range (2016-06-04 - 2016-06-10) has too much cloud cover (100.00%).
Image for point (24.090357, 65.925954) and date range (2016-07-04 - 2016-07-10) has too much cloud cover (100.00%).
Image for point (24.090357, 65.925954) and date range (2017-03-18 - 2017-03-24) has too much cloud cover (100.00%).
Image for point (24.090357, 65.925954) and date range (2017-04-04 - 2017-04-10) has too much cloud cover (100.00%).
Image for point (24.090357, 65.925954) and date range (2017-05-04 - 2017-05-10) has too much cloud cover (100.00%).
Image for point (24.090357, 65.925954) and date range (2017-06-04 - 2017-

  6%|▌         | 2/36 [01:35<27:10, 47.96s/it]

...

Image for point (20.513277, 67.855095) and date range (2025-07-04 - 2025-07-10) has too much cloud cover (100.00%).
Image for point (20.493279, 67.864071) and date range (2016-03-18 - 2016-03-24) has too much cloud cover (79.51%).
Image for point (20.493279, 67.864071) and date range (2016-04-04 - 2016-04-10) has too much cloud cover (73.49%).
Image for point (20.493279, 67.864071) and date range (2016-05-04 - 2016-05-10) has too much cloud cover (80.00%).
Image for point (20.493279, 67.864071) and date range (2016-05-18 - 2016-05-24) has too much cloud cover (100.00%).
Image for point (20.493279, 67.864071) and date range (2016-06-04 - 2016-06-10) has too much cloud cover (33.13%).
Image for point (20.493279, 67.864071) and date range (2016-07-04 - 2016-07-10) has too much cloud cover (100.00%).
Image for point (20.493279, 67.864071) and date range (2017-04-18 - 2017-04-24) has too much cloud cover (100.00%).
Image for point (20.493279, 67.864071) and date range (2017-05-04 - 2017-05-

 97%|█████████▋| 35/36 [25:38<00:42, 42.68s/it]

Image for point (20.493279, 67.864071) and date range (2025-07-04 - 2025-07-10) has too much cloud cover (100.00%).
Image for point (20.36037, 67.944237) and date range (2016-03-18 - 2016-03-24) has too much cloud cover (100.00%).
Image for point (20.36037, 67.944237) and date range (2016-04-04 - 2016-04-10) has too much cloud cover (100.00%).
Image for point (20.36037, 67.944237) and date range (2016-05-04 - 2016-05-10) has too much cloud cover (100.00%).
Image for point (20.36037, 67.944237) and date range (2016-05-18 - 2016-05-24) has too much cloud cover (99.53%).
Image for point (20.36037, 67.944237) and date range (2016-06-04 - 2016-06-10) has too much cloud cover (16.38%).
Image for point (20.36037, 67.944237) and date range (2016-07-04 - 2016-07-10) has too much cloud cover (100.00%).
Image for point (20.36037, 67.944237) and date range (2017-04-18 - 2017-04-24) has too much cloud cover (100.00%).
Image for point (20.36037, 67.944237) and date range (2017-05-04 - 2017-05-10) ha

100%|██████████| 36/36 [26:20<00:00, 43.89s/it]

Image for point (20.36037, 67.944237) and date range (2025-07-04 - 2025-07-10) has too much cloud cover (100.00%).


In [17]:
points_of_interest = pd.read_csv("data/points_of_interest_10m.csv").values
points_of_interest

array([[24.13044 , 65.918934],
       [24.090357, 65.925954],
       [24.037013, 65.965172],
       [23.718624, 66.199689],
       [23.717337, 66.2156  ],
       [23.691974, 66.237346],
       [23.687682, 66.254443],
       [23.647084, 66.28408 ],
       [23.661933, 66.303059],
       [23.652706, 66.328728],
       [23.678198, 66.375395],
       [23.649359, 66.395818],
       [23.671031, 66.461546],
       [23.735018, 66.503279],
       [23.800936, 66.543763],
       [23.859816, 66.575277],
       [23.877926, 66.611743],
       [23.863077, 66.650826],
       [23.646011, 67.107207],
       [23.550525, 67.17917 ],
       [23.503618, 67.19313 ],
       [23.408046, 67.204207],
       [23.259301, 67.25573 ],
       [22.771912, 67.263743],
       [22.506223, 67.430731],
       [22.353144, 67.48826 ],
       [21.686883, 67.680168],
       [21.660275, 67.678913],
       [21.633239, 67.678179],
       [21.621695, 67.69475 ],
       [21.22992 , 67.731223],
       [20.64992 , 67.84291 ],
       [

In [ ]:
def display_last_three_years(light_up=1):
    import matplotlib.pyplot as plt
    import numpy as np
    import os
    from PIL import Image
    
    total_points = len(points_of_interest)
    total_months = len(months_range)
    
    # Get the last two years
    last_three_years = list(years_range)[-3:]
    total_cols = len(last_three_years) * total_months  # 3 years * months per year
    
    # Create grid: rows = points, columns = months across 3 years
    fig, axes = plt.subplots(total_points, total_cols, 
                             figsize=(total_cols * 2, total_points * 3))
    
    # Handle different array shapes
    if total_points == 1 and total_cols == 1:
        axes = np.array([[axes]])
    elif total_points == 1:
        axes = axes.reshape(1, -1)
    elif total_cols == 1:
        axes = axes.reshape(-1, 1)
    
    for index, point in enumerate(points_of_interest):
        col_idx = 0
        
        for year in last_three_years:
            for month_idx, month in enumerate(months_range):
                save_path = save_path_for_point(index, point[0], point[1], month, year)
                save_path_cloudy = save_path.replace(".png", "_C.png")
                ax = axes[index, col_idx]
                if os.path.exists(save_path_cloudy):
                    img = Image.open(save_path_cloudy)
                    ax.imshow(img)
                    ax.set_title(f"{year}-{month:02d} (Cloudy)", fontsize=9)
                elif os.path.exists(save_path):
                    img = Image.open(save_path)
                    img = np.array(img)
                    if month > 4:
                        img = np.clip(img * 2, 0, 255).astype(np.uint8)
                    img = Image.fromarray(img)
                    ax.imshow(img)
                    ax.set_title(f"{year}-{month:02d}", fontsize=9)
                else:
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=8)
                    ax.set_title(f"{year}-{month:02d}", fontsize=9)
                
                ax.axis('off')
                
                col_idx += 1
        
        # Add row label
        axes[index, 0].set_ylabel(f"Point {index}\n({point[0]:.2f}, {point[1]:.2f})", 
                                  fontsize=10, rotation=0, labelpad=60, va='center')
    
    plt.suptitle(f'Last Three Years: {last_three_years[0]} - {last_three_years[-1]}', 
                 fontsize=14, y=0.998)
    plt.tight_layout()
    plt.show()

# Call the function
display_last_three_years()

In [64]:
date_start = "2020-02-01"
date_end = "2020-02-05"
img = download_data_for_point(points_of_interest[0][0], points_of_interest[0][1], date_start, date_end)


In [65]:
img.shape

(144, 147, 3)

In [88]:
# read data for points of interest read csv file with points of interest
import pandas as pd


df = pd.read_csv("data/points_of_interest_10m.csv")
points_of_interest = df.values.tolist()

In [103]:
delta_bbox_x,delta_bbox_y = 0.0175 , 0.006
center = (24.143529,65.853763)
betsiboka_coords_wgs84 = (center[0] - delta_bbox_x, center[1] - delta_bbox_y, center[0] + delta_bbox_x, center[1] + delta_bbox_y)
betsiboka_bbox = BBox(bbox = betsiboka_coords_wgs84, crs=CRS.WGS84)

resolution = 10
betsiboka_bbox = BBox(bbox=betsiboka_coords_wgs84, crs=CRS.WGS84)
betsiboka_size = bbox_to_dimensions(betsiboka_bbox, resolution=resolution)

print(f"Image shape at {resolution} m resolution: {betsiboka_size} pixels")



evalscript_true_color = """
    //VERSION=3

    function setup() {
        return {
            input: [{
                bands: ["B02", "B03", "B04"]
            }],
            output: {
                bands: 3
            }
        };
    }

    function evaluatePixel(sample) {
        return [sample.B04, sample.B03, sample.B02];
    }
"""

request_true_color = SentinelHubRequest(
    evalscript=evalscript_true_color,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L1C,
            time_interval=("2020-06-01", "2020-06-30"),
            mosaicking_order=MosaickingOrder.LEAST_CC
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.PNG)],
    bbox=betsiboka_bbox,
    size=betsiboka_size,
    config=config,
    data_folder="data/betsiboka_2020_02"
)
true_color_imgs = request_true_color.get_data(save_data=True)
image = true_color_imgs[0]
print(f"Image type: {image.dtype}")

# plot function
# factor 1/255 to scale between 0-1
# factor 3.5 to increase brightness
plot_image(image, factor=3.5 / 255)



Image shape at 10 m resolution: (166, 126) pixels


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.08235294117647059..1.66078431372549].


Image type: uint8
